In [16]:
run_numbers = [564445,564430,564414,564400,564387,564374,564373,564359,564356]
run_labels = ['all'] + run_numbers
root_path = '/home/wuct/MetaData/MC/OO/cent'
categories = ['sel8_GoodzVtx_nopileup', 'sel8_GoodzVtx', 'sel8', 'vtxtrigger'] 
histo_names = ['hf-candidate-creator-2prong/hSelCollisionsCent', 'hf-candidate-creator-2prong/hCollisions']
histo_labels = ['cent', 'evt sels']

In [17]:
import pathlib as PATH
from collections import defaultdict
merged_file_paths = {category: None for category in categories}
for category in categories:
    merged_file_paths[category] = PATH.Path(root_path) / category / 'AnalysisResults.root'
    print(f'{category}: {merged_file_paths[category]}')
runs_file_paths = defaultdict(dict)
for category in categories:
    for run_number in run_numbers:
        file_path = PATH.Path(root_path) / category / f'{run_number}/0001' / f'AnalysisResults.root'
        runs_file_paths[category][run_number] = file_path
        print(f'{category} - {run_number}: {file_path}')

sel8_GoodzVtx_nopileup: /home/wuct/MetaData/MC/OO/cent/sel8_GoodzVtx_nopileup/AnalysisResults.root
sel8_GoodzVtx: /home/wuct/MetaData/MC/OO/cent/sel8_GoodzVtx/AnalysisResults.root
sel8: /home/wuct/MetaData/MC/OO/cent/sel8/AnalysisResults.root
vtxtrigger: /home/wuct/MetaData/MC/OO/cent/vtxtrigger/AnalysisResults.root
sel8_GoodzVtx_nopileup - 564445: /home/wuct/MetaData/MC/OO/cent/sel8_GoodzVtx_nopileup/564445/0001/AnalysisResults.root
sel8_GoodzVtx_nopileup - 564430: /home/wuct/MetaData/MC/OO/cent/sel8_GoodzVtx_nopileup/564430/0001/AnalysisResults.root
sel8_GoodzVtx_nopileup - 564414: /home/wuct/MetaData/MC/OO/cent/sel8_GoodzVtx_nopileup/564414/0001/AnalysisResults.root
sel8_GoodzVtx_nopileup - 564400: /home/wuct/MetaData/MC/OO/cent/sel8_GoodzVtx_nopileup/564400/0001/AnalysisResults.root
sel8_GoodzVtx_nopileup - 564387: /home/wuct/MetaData/MC/OO/cent/sel8_GoodzVtx_nopileup/564387/0001/AnalysisResults.root
sel8_GoodzVtx_nopileup - 564374: /home/wuct/MetaData/MC/OO/cent/sel8_GoodzVtx_nopi

In [18]:
import ROOT
def get_histo():
    histos = defaultdict(lambda: defaultdict(dict))
    histos = {histo_label: {category: {label: None for label in run_labels} for category in categories} for histo_label in histo_labels}
    for category in categories:
        merged_file = ROOT.TFile.Open(str(merged_file_paths[category]))
        for  histo_name, histo_label in zip(histo_names, histo_labels):
            merged_histo = merged_file.Get(histo_name)
            merged_histo.SetDirectory(0)
            histos[histo_label][category]['all'] = merged_histo
        for run_number in run_numbers:
            run_file = ROOT.TFile.Open(str(runs_file_paths[category][run_number]))
            for histo_name, histo_label in zip(histo_names, histo_labels):
                run_histo = run_file.Get(histo_name)
                run_histo.SetDirectory(0)
                histos[histo_label][category][run_number] = run_histo
    return histos

In [7]:
histos = get_histo()
color_palette = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen+2, ROOT.kMagenta]
cans = []
legs = []
ROOT.gStyle.SetOptStat(0)
for histo_label in histo_labels:
    cans.append(ROOT.TCanvas(f'canvas_{histo_label}', f'canvas_{histo_label}', 800, 800))
    cans[-1].SetLeftMargin(0.15)
    cans[-1].SetRightMargin(0.05)
    cans[-1].SetBottomMargin(0.1)
    cans[-1].SetTopMargin(0.1)
    legs.append(ROOT.TLegend(0.45, 0.65, 0.95, 0.90))
    legs[-1].SetFillStyle(0)
    legs[-1].SetTextSize(0.04)
    y_max = 0
    y_min = 0
    for i, category in enumerate(categories):
        histo = histos[histo_label][category]['all']
        histo.SetLineColor(color_palette[i])
        histo.SetMarkerColor(color_palette[i])
        histo.SetLineWidth(2)
        # histo.SetMarkerStyle(20)
        # histo.SetMarkerSize(1.5)
        if i == 0:
            if histo_label == 'cent':
                histo.GetXaxis().SetTitle('Centrality all runs')
                histo.Scale(1.0 / histo.Integral())
                y_max = max([histos[histo_label][cat]['all'].GetMaximum() / histos[histo_label][cat]['all'].Integral() for cat in categories])
                y_min = min([histos[histo_label][cat]['all'].GetMinimum() / histos[histo_label][cat]['all'].Integral() for cat in categories])
                histo.GetYaxis().SetTitle(f'Normalized {histo.GetYaxis().GetTitle()}')
            elif histo_label == 'evt sels':
                histo.GetXaxis().SetTitle('Event selection steps')
                y_max = max([histos[histo_label][cat]['all'].GetMaximum() for cat in categories])
                y_min = min([histos[histo_label][cat]['all'].GetMinimum() for cat in categories])
            histo.GetYaxis().SetRangeUser(y_min * 0.8, y_max * 1.2)
            histo.Draw('hist')
        else:
            if histo_label == 'cent':
                histo.Scale(1.0 / histo.Integral())
            histo.Draw('hist SAME')
        legs[-1].AddEntry(histo, category, 'l')
    legs[-1].Draw()
    cans[-1].SaveAs(f'{histo_label}_merged.png')

Info in <TCanvas::Print>: png file cent_merged.png has been created
Info in <TCanvas::Print>: png file evt sels_merged.png has been created


In [44]:
# compare cent distributions for each run
histos = get_histo()
color_palette = [ROOT.kAzure, ROOT.kBlue, ROOT.kGreen+2, ROOT.kMagenta, ROOT.kCyan, ROOT.kOrange, ROOT.kViolet, ROOT.kPink, ROOT.kBlack]

for histo_label in histo_labels:
    if histo_label == 'cent':
        y_max = max([histos[histo_label][category][run_number].GetMaximum() / histos[histo_label][category][run_number].Integral() for category in categories for run_number in run_numbers])
        y_min = min([histos[histo_label][category][run_number].GetMinimum() / histos[histo_label][category][run_number].Integral() for category in categories for run_number in run_numbers])
    else:
        y_max = max([histos[histo_label][category][run_number].GetMaximum() / histos[histo_label][category][run_number].GetBinContent(1) for category in categories for run_number in run_numbers])
        y_min = min([histos[histo_label][category][run_number].GetMinimum() / histos[histo_label][category][run_number].GetBinContent(1) for category in categories for run_number in run_numbers])
    for category in categories:
        cans.append(ROOT.TCanvas(f'canvas_{histo_label}_{category}', f'canvas_{histo_label}_{category}', 800, 800))
        cans[-1].SetLeftMargin(0.15)
        cans[-1].SetRightMargin(0.05)
        cans[-1].SetBottomMargin(0.1)
        cans[-1].SetTopMargin(0.1)
        legs.append(ROOT.TLegend(0.75, 0.55, 0.95, 0.90))
        legs[-1].SetTextSize(0.04)
        # y_max = 0
        # y_min = 0
        for i, run_number in enumerate(run_numbers):
            histo = histos[histo_label][category][run_number]
            histo.SetLineColor(color_palette[i])
            histo.SetMarkerColor(color_palette[i])
            histo.SetLineWidth(2)
            # histo.SetMarkerStyle(20)
            # histo.SetMarkerSize(1.5)
            if i == 0:
                if histo_label == 'cent':
                    histo.GetXaxis().SetTitle(f'Centrality {category}')
                    histo.Scale(1.0 / histo.Integral())
                    histo.GetYaxis().SetTitle(f'Normalized {histo.GetYaxis().GetTitle()}')
                    histo.GetYaxis().SetRangeUser(y_min * 0.8, y_max * 1.2)
                elif histo_label == 'evt sels':
                    histo.GetXaxis().SetTitle('Event selection steps')
                    histo.SetTitle(f'{histo.GetTitle()} normalized to the 1st bin content')
                    # y_max = 1.0 / histo.GetBinContent(1) / histo.Integral() * 1.2
                    # y_min = 1.0 / histo.GetBinContent(1) / histo.Integral() * 0.8
                    histo.Scale(1.0 / histo.GetBinContent(1))
                    histo.GetYaxis().SetRangeUser(y_min * 0.9, y_max * 1.5)
                    # cans[-1].SetLogy()
                
                histo.Draw('hist')
            else:
                if histo_label == 'cent':
                    histo.Scale(1.0 / histo.Integral())
                else:
                    histo.Scale(1.0 / histo.GetBinContent(1))
                histo.Draw('hist SAME')
            legs[-1].AddEntry(histo, str(run_number), 'l')
        legs[-1].Draw()
        cans[-1].SaveAs(f'{histo_label}_{category}_runs.png')

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_cent_sel8_GoodzVtx_nopileup
Info in <TCanvas::Print>: png file cent_sel8_GoodzVtx_nopileup_runs.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_cent_sel8_GoodzVtx
Info in <TCanvas::Print>: png file cent_sel8_GoodzVtx_runs.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_cent_sel8
Info in <TCanvas::Print>: png file cent_sel8_runs.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_cent_vtxtrigger
Info in <TCanvas::Print>: png file cent_vtxtrigger_runs.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_evt sels_sel8_GoodzVtx_nopileup
Info in <TCanvas::Print>: png file evt sels_sel8_GoodzVtx_nopileup_runs.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_evt sels_sel8_GoodzVtx
Info in <TCanvas::

In [6]:
# compare the different categories for each run
histos = get_histo()
color_palette = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen+2, ROOT.kMagenta]
cans = []
legs = []
ROOT.gStyle.SetOptStat(0)
for histo_label in histo_labels:
    if histo_label == 'cent':
        y_max = max([histos[histo_label][category][run_number].GetMaximum() / histos[histo_label][category][run_number].Integral() for category in categories for run_number in run_numbers])
        y_min = min([histos[histo_label][category][run_number].GetMinimum() / histos[histo_label][category][run_number].Integral() for category in categories for run_number in run_numbers])
    else:
        y_max = max([histos[histo_label][category][run_number].GetMaximum() / histos[histo_label][category][run_number].GetBinContent(1) for category in categories for run_number in run_numbers])
        y_min = min([histos[histo_label][category][run_number].GetMinimum() / histos[histo_label][category][run_number].GetBinContent(1) for category in categories for run_number in run_numbers])
    for run_number in run_numbers:
        cans.append(ROOT.TCanvas(f'canvas_{histo_label}_run{run_number}', f'canvas_{histo_label}_run{run_number}', 800, 800))
        cans[-1].SetLeftMargin(0.15)
        cans[-1].SetRightMargin(0.05)
        cans[-1].SetBottomMargin(0.1)
        cans[-1].SetTopMargin(0.1)
        legs.append(ROOT.TLegend(0.45, 0.65, 0.95, 0.90))
        legs[-1].SetFillStyle(0)
        legs[-1].SetTextSize(0.04)
        # y_max = 0
        # y_min = 0
        for i, category in enumerate(categories):
            histo = histos[histo_label][category][run_number]
            histo.SetLineColor(color_palette[i])
            histo.SetMarkerColor(color_palette[i])
            histo.SetLineWidth(2)
            # histo.SetMarkerStyle(20)
            # histo.SetMarkerSize(1.5)
            if i == 0:
                if histo_label == 'cent':
                    histo.GetXaxis().SetTitle(f'Centrality run {run_number}')
                    histo.GetYaxis().SetTitle(f'Normalized {histo.GetYaxis().GetTitle()}')
                    histo.Scale(1.0 / histo.Integral())
                    histo.GetYaxis().SetRangeUser(y_min * 0.8, y_max * 1.2)
                elif histo_label == 'evt sels':
                    histo.GetXaxis().SetTitle(f'Event selection steps run {run_number}')
                    histo.SetTitle(f'{histo.GetTitle()} normalized to the 1st bin content')
                    # y_max = 1.0 / histo.GetBinContent(1) / histo.Integral() * 1.2
                    # y_min = 1.0 / histo.GetBinContent(1) / histo.Integral() * 0.8
                    histo.Scale(1.0 / histo.GetBinContent(1))
                    histo.GetYaxis().SetRangeUser(y_min * 0.9, y_max * 1.5)
                    # cans[-1].SetLogy()
                histo.Draw('hist')
            else:
                if histo_label == 'cent':
                    histo.Scale(1.0 / histo.Integral())
                else:
                    histo.Scale(1.0 / histo.GetBinContent(1))
                histo.Draw('hist SAME')
            legs[-1].AddEntry(histo, category, 'l')
        legs[-1].Draw()
        cans[-1].SaveAs(f'{histo_label}_run{run_number}_categories.png')

Info in <TCanvas::Print>: png file cent_run564445_categories.png has been created
Info in <TCanvas::Print>: png file cent_run564430_categories.png has been created
Info in <TCanvas::Print>: png file cent_run564414_categories.png has been created
Info in <TCanvas::Print>: png file cent_run564400_categories.png has been created
Info in <TCanvas::Print>: png file cent_run564387_categories.png has been created
Info in <TCanvas::Print>: png file cent_run564374_categories.png has been created
Info in <TCanvas::Print>: png file cent_run564373_categories.png has been created
Info in <TCanvas::Print>: png file cent_run564359_categories.png has been created
Info in <TCanvas::Print>: png file cent_run564356_categories.png has been created
Info in <TCanvas::Print>: png file evt sels_run564445_categories.png has been created
Info in <TCanvas::Print>: png file evt sels_run564430_categories.png has been created
Info in <TCanvas::Print>: png file evt sels_run564414_categories.png has been created
Info

In [24]:
# event number vs run number for each category
histos = get_histo()
# evts = defaultdict(dict)
can = []
can.append(ROOT.TCanvas('canvas_evts', 'canvas_evts', 800, 800))
can[0].SetLeftMargin(0.15)
can[0].SetRightMargin(0.05)
can[0].SetBottomMargin(0.1)
can[0].SetTopMargin(0.1)
ROOT.gStyle.SetOptStat(0)

legs=[]
legs.append(ROOT.TLegend(0.45, 0.65, 0.95, 0.90))
legs[-1].SetFillStyle(0)
legs[-1].SetTextSize(0.04)

evts_histos = []
color_palette = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen+2, ROOT.kMagenta]
run_numbers.reverse()
for i, category in enumerate(categories):
    evts_histos.append(ROOT.TH1D(' ', f'evts_{category}', len(run_numbers), 0, len(run_numbers)))
    for run_number in run_numbers:
        evt_number = histos['cent'][category][run_number].Integral()
        print(f'{category} - {run_number}: {evt_number}')
        evts_histos[-1].SetBinContent(run_numbers.index(run_number)+1, evt_number)
        evts_histos[-1].GetXaxis().SetBinLabel(run_numbers.index(run_number)+1, str(run_number))
    evts_histos[-1].SetLineColor(color_palette[i])
    evts_histos[-1].SetMarkerColor(color_palette[i])
    legs[-1].AddEntry(evts_histos[-1], category, 'l')
can[0].cd()
y_max = max([histo.GetMaximum() for histo in evts_histos])
y_min = min([histo.GetMinimum() for histo in evts_histos])
for i, histo in enumerate(evts_histos):
    if i == 0:
        histo.GetYaxis().SetTitle('Number of events')
        histo.GetXaxis().SetTitle('Run number')
        histo.GetYaxis().SetRangeUser(y_min * 0.6, y_max * 14)
        histo.Draw('hist')
        can[0].SetLogy()
    else:
        histo.Draw('hist SAME')
legs[-1].Draw()
can[0].SaveAs('events_per_run.png')
# for category in categories:
#     evts_histos.append(ROOT.TH1D(f'evts_{category}', f'evts_{category}', {run_number: run_number for run_number in run_numbers}, min(run_numbers)-0.5, max(run_numbers)+0.5))
#     for run_number in run_numbers:
#         histo = histos['evt sels'][category][run_number]
#         evt_number = histo.GetBinContent(-1)
#         print(f'{category} - {run_number}: {evt_number}')
#         evts_histos[-1].SetBinContent(run_numbers.index(run_number)+1, evt_number)
#     evts_histos[-1].SetLineColor(color_palette[categories.index(category)])
#     evts_histos[-1].SetMarkerColor(color_palette[categories.index(category)])
#     legs[-1].AddEntry(evts_histos[-1], category, 'l')
# can[0].cd()
# for histo in evts_histos:
#     histo.Draw('hist SAME')
# legs[-1].Draw()
# evts_histos[-1].Draw('hist')
# can[0].SaveAs('events_per_run.png')

sel8_GoodzVtx_nopileup - 564356: 1728275.0
sel8_GoodzVtx_nopileup - 564359: 5162027.0
sel8_GoodzVtx_nopileup - 564373: 509509.0
sel8_GoodzVtx_nopileup - 564374: 13267979.0
sel8_GoodzVtx_nopileup - 564387: 15162905.0
sel8_GoodzVtx_nopileup - 564400: 17557336.0
sel8_GoodzVtx_nopileup - 564414: 13554504.0
sel8_GoodzVtx_nopileup - 564430: 20536985.0
sel8_GoodzVtx_nopileup - 564445: 11978033.0
sel8_GoodzVtx - 564356: 1741001.0
sel8_GoodzVtx - 564359: 5215056.0
sel8_GoodzVtx - 564373: 519940.0
sel8_GoodzVtx - 564374: 15845835.0
sel8_GoodzVtx - 564387: 15435560.0
sel8_GoodzVtx - 564400: 17000047.0
sel8_GoodzVtx - 564414: 13356145.0
sel8_GoodzVtx - 564430: 20912653.0
sel8_GoodzVtx - 564445: 12886893.0
sel8 - 564356: 1817876.0
sel8 - 564359: 5457947.0
sel8 - 564373: 799888.0
sel8 - 564374: 16738690.0
sel8 - 564387: 17965669.0
sel8 - 564400: 23792487.0
sel8 - 564414: 16723053.0
sel8 - 564430: 24388491.0
sel8 - 564445: 15598828.0
vtxtrigger - 564356: 1963348.0
vtxtrigger - 564359: 5892857.0
vtxtr

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_evts
Warning in <TROOT::Append>: Replacing existing TH1:   (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1:   (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1:   (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1:   (Potential memory leak).
Info in <TCanvas::Print>: png file events_per_run.png has been created
